In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data.yaml
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/PublicDataset01019.txt
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/WEB08651.txt
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/WEB07304.txt
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/WEB07388.txt
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/WEB02204.txt
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/WEB04894.txt
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/AoF05996.txt
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/WEB04652.txt
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/AoF02633.txt
/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels/PublicDataset00006.tx

In [2]:
!pip install -q -U ultralytics
import ultralytics
print("Ultralytics version:", ultralytics.__version__)
!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics version: 8.4.138
Wed Sep  2 06:11:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf        

In [3]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')

input/
  datasets/
    sayedgamal99/
      smoke-fire-detection-yolo/
        data/
          val/
            labels/
            images/
          test/
            labels/
            images/
          train/
            labels/
            images/


In [4]:
DATA_YAML_ORIGINAL = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data.yaml"

with open(DATA_YAML_ORIGINAL) as f:
    print(f.read())

path: /kaggle/working/D Fire Dataset  # dataset root dir
train: data/train/images  # train images (relative to 'path')
val: data/val/images  # val images (relative to 'path')
test: data/test/images  # test images (relative to 'path')

# Classes
names: ['smoke', 'fire']  # Replace with your actual class names

# Counts
nc: 2  # number of classes
train_count: 14122
val_count: 3099
test_count: 4306



In [5]:
BASE = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data"

corrected_yaml = f"""
train: {BASE}/train/images
val: {BASE}/val/images
test: {BASE}/test/images
nc: 2
names: ['smoke', 'fire']
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(corrected_yaml)

DATA_YAML = "/kaggle/working/data.yaml"
print("Using corrected data.yaml:")
with open(DATA_YAML) as f:
    print(f.read())

Using corrected data.yaml:

train: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/train/images
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images
test: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images
nc: 2
names: ['smoke', 'fire']



In [6]:
from ultralytics import YOLO

yolo11_30ep_model = YOLO("yolo11n.pt")

yolo11_30ep_results = yolo11_30ep_model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=32,
    device=[0, 1],
    project="/kaggle/working/runs",
    name="yolo11n_30ep",
)

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, 

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      2.52G      1.754      1.952      1.578         31        640: 100% ━━━━━━━━━━━━ 441/441 3.8it/s 1:55
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 4.9it/s 10.1s
                   all       3094       3917      0.489       0.41      0.396      0.168

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      2.52G      1.747      1.861      1.572         33        640: 100% ━━━━━━━━━━━━ 441/441 3.9it/s 1:54
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 4.8it/s 10.1s
                   all       3094       3917      0.386      0.412      0.347      0.144

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      2.52G      1.738      1.801      1.563         14        640: 100% ━━━━━━━━━

In [7]:
yolo11_30ep_best = "/kaggle/working/runs/yolo11n_30ep/weights/best.pt"
yolo11_30ep_val_model = YOLO(yolo11_30ep_best)
yolo11_30ep_metrics = yolo11_30ep_val_model.val(data=DATA_YAML)

print("YOLOv11n (30 epochs) — mAP50-95:", yolo11_30ep_metrics.box.map)
print("YOLOv11n (30 epochs) — mAP50:", yolo11_30ep_metrics.box.map50)
print("Precision:", yolo11_30ep_metrics.box.mp)
print("Recall:", yolo11_30ep_metrics.box.mr)

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 233.2±213.1 MB/s, size: 122.4 KB)
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels... 3094 images, 1375 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 3099/3099 1.2Kit/s 2.5s
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-

In [8]:
import glob, time

val_images = glob.glob(f"{BASE}/val/images/*.jpg")[:50]

start = time.time()
for img_path in val_images:
    _ = yolo11_30ep_val_model.predict(img_path, verbose=False)
infer_ms = ((time.time() - start) / len(val_images)) * 1000
print(f"YOLOv11n (30 epochs) average inference time: {infer_ms:.2f} ms/frame")

YOLOv11n (30 epochs) average inference time: 19.24 ms/frame


In [9]:
import shutil
shutil.make_archive("/kaggle/working/yolo11n_30ep_results", 'zip', "/kaggle/working/runs/yolo11n_30ep")

from IPython.display import FileLink
FileLink("/kaggle/working/yolo11n_30ep_results.zip")

/kaggle/working/yolo11n_30ep_results.zip

In [10]:
# from ultralytics import YOLO

# yolo11_model = YOLO("yolo11n.pt")

# yolo11_results = yolo11_model.train(
#     data=DATA_YAML,
#     epochs=15,
#     imgsz=640,
#     batch=32,          # bumped up since we're using 2 GPUs
#     device=[0, 1],      # use both T4s
#     project="/kaggle/working/runs",
#     name="yolo11n_finetuned",
# )

In [11]:
# from ultralytics import YOLO

# yolo11_model = YOLO("yolo11n.pt")

# yolo11_results = yolo11_model.train(
#     data=DATA_YAML,
#     epochs=15,
#     imgsz=640,
#     batch=32,          # bumped up since we're using 2 GPUs
#     device=[0, 1],      # use both T4s
#     project="/kaggle/working/runs",
#     name="yolo11n_finetuned",
# )

In [12]:
# from ultralytics import RTDETR

# rtdetr_model = RTDETR("rtdetr-l.pt")

# rtdetr_results = rtdetr_model.train(
#     data=DATA_YAML,
#     epochs=15,
#     imgsz=640,
#     batch=16,           # RT-DETR is heavier per-image than YOLO, kept lower than YOLO's batch
#     device=[0, 1],
#     project="/kaggle/working/runs",
#     name="rtdetr_l_finetuned",
# )

In [13]:
# rtdetr_best = "/kaggle/working/runs/rtdetr_l_finetuned/weights/best.pt"
# rtdetr_val_model = RTDETR(rtdetr_best)
# rtdetr_metrics = rtdetr_val_model.val(data=DATA_YAML)

# print("RT-DETR-L — mAP50-95:", rtdetr_metrics.box.map)
# print("RT-DETR-L — mAP50:", rtdetr_metrics.box.map50)

In [14]:
# from ultralytics import YOLO

# yolo11_best = "/kaggle/working/runs/yolo11n_finetuned/weights/best.pt"

# yolo11_val_model = YOLO(yolo11_best)

# yolo11_metrics = yolo11_val_model.val(data=DATA_YAML)

# print("YOLOv11n — mAP50-95:", yolo11_metrics.box.map)
# print("YOLOv11n — mAP50:", yolo11_metrics.box.map50)

In [15]:
# from ultralytics import RTDETR

# rtdetr_best = "/kaggle/working/runs/rtdetr_l_finetuned/weights/best.pt"

# rtdetr_val_model = RTDETR(rtdetr_best)

# rtdetr_metrics = rtdetr_val_model.val(data=DATA_YAML)

# print("RT-DETR-L — mAP50-95:", rtdetr_metrics.box.map)
# print("RT-DETR-L — mAP50:", rtdetr_metrics.box.map50)

In [16]:
# comparison_md = f"""# Model Comparison — YOLOv11 vs RT-DETR-L

# | Model | mAP50-95 | mAP50 |
# |---|---|---|
# | YOLOv11n | {yolo11_metrics.box.map:.4f} | {yolo11_metrics.box.map50:.4f} |
# | RT-DETR-L | {rtdetr_metrics.box.map:.4f} | {rtdetr_metrics.box.map50:.4f} |
# """

# with open("/kaggle/working/MODEL_COMPARISON_KAGGLE.md", "w") as f:
#     f.write(comparison_md)

# print(comparison_md)

In [17]:
# from ultralytics import YOLO

# yolov8_model = YOLO("yolov8n.pt")

# yolov8_results = yolov8_model.train(
#     data=DATA_YAML,
#     epochs=15,
#     imgsz=640,
#     batch=32,
#     device=[0, 1],
#     project="/kaggle/working/runs",
#     name="yolov8n_finetuned"
# )

In [18]:
# from ultralytics import YOLO

# yolo26_model = YOLO("yolo26n.pt")

# yolo26_results = yolo26_model.train(
#     data=DATA_YAML,
#     epochs=15,
#     imgsz=640,
#     batch=32,
#     device=[0, 1],
#     project="/kaggle/working/runs",
#     name="yolo26n_finetuned"
# )

In [19]:
# from ultralytics import YOLO

# yolov8_best = "/kaggle/working/runs/yolov8n_finetuned/weights/best.pt"

# yolov8_val_model = YOLO(yolov8_best)

# yolov8_metrics = yolov8_val_model.val(data=DATA_YAML)

# print("YOLOv8n mAP50-95:", yolov8_metrics.box.map)
# print("YOLOv8n mAP50:", yolov8_metrics.box.map50)

In [20]:
# from ultralytics import YOLO

# yolo26_best = "/kaggle/working/runs/yolo26n_finetuned/weights/best.pt"

# yolo26_val_model = YOLO(yolo26_best)

# yolo26_metrics = yolo26_val_model.val(data=DATA_YAML)

# print("YOLO26n mAP50-95:", yolo26_metrics.box.map)
# print("YOLO26n mAP50:", yolo26_metrics.box.map50)

In [21]:
# comparison_md = f"""# Model Comparison

# | Model | mAP50-95 | mAP50 |
# |---|---|---|
# | YOLOv8n | {yolov8_metrics.box.map:.4f} | {yolov8_metrics.box.map50:.4f} |
# | YOLOv11n | {yolo11_metrics.box.map:.4f} | {yolo11_metrics.box.map50:.4f} |
# | YOLO26n | {yolo26_metrics.box.map:.4f} | {yolo26_metrics.box.map50:.4f} |
# | RT-DETR-L | {rtdetr_metrics.box.map:.4f} | {rtdetr_metrics.box.map50:.4f} |
# """

# print(comparison_md)